# Step 11.5 — Weighted Value Retrieval：Attention Weights 最后怎样真正取回信息？

Step 11.4 已经得到 normalized attention weights：

$$
A
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
\in
\mathbb R^{B\times T\times T}.
$$

其中 `A[b,i,j]` 表示：sample `b` 中 Query position `i` 给 Key position `j` 的权重。

但是 A 本身只回答：

> **“从哪些 positions 取信息，各取多少？”**

真正要被取回的内容存放在 Value tensor：

$$
V\in\mathbb R^{B\times T\times d_v}.
$$

所以最后一步就是：

$$
\boxed{O=AV}.
$$

本节会完成一个完整 tiny single-head calculation；到这里停止，不进入 Step 12.1。


## 1. 先只看一个 Query：一行 Attention Weights 怎样混合 Values？

假设某个 Query position `i` 对三个 Key positions 的 attention weights 是：

$$
A_{i,:}
=
[0.2,0.5,0.3].
$$

对应三个 Value vectors：

$$
v_0=[1,0],
\qquad
v_1=[0,2],
\qquad
v_2=[3,1].
$$

那么当前 Query 的输出 representation：

$$
o_i
=
0.2v_0+0.5v_1+0.3v_2.
$$

逐项：

$$
0.2[1,0]=[0.2,0],
$$

$$
0.5[0,2]=[0,1],
$$

$$
0.3[3,1]=[0.9,0.3].
$$

相加：

$$
\boxed{o_i=[1.1,1.3]}.
$$

这和 Step 10.4 的 weighted context 思想完全一致，但现在它已经放进 Self-Attention 的 Q/K/V 角色体系里：

```text
Q/K -> 决定 weights
V   -> 提供 payload
A   -> 决定每个 payload 混合多少
```

所以 Attention 的输出不是“最大的 Key”，而是 Value vectors 的 weighted mixture（加权混合）。


## 2. 所有 Query Positions 一次计算：为什么正好是 `A @ V`？

Batch 下：

$$
A:(B,T,T)
$$

而：

$$
V:(B,T,d_v).
$$

矩阵乘法：

$$
\boxed{
(B,T,T)
@
(B,T,d_v)
\rightarrow
(B,T,d_v)
}
$$

定义输出：

$$
O=AV
\in
\mathbb R^{B\times T\times d_v}.
$$

其中：

$$
\boxed{
o_i
=
\sum_{j=0}^{T-1}
A_{i,j}v_j
}
$$

所以：

```text
O 的第 i 行
= Query position i
  根据自己的 attention distribution
  对所有 Value rows 做 weighted sum 后的结果
```

注意输出 feature width 是 `d_v`，不是 `d_k`。

因为 `d_k` 是 Query/Key matching space 的 width，而真正被混合的 payload 来自 V，所以最终 retrieval output 的 width 跟随 `d_v`。


## 3. 完整 Tiny Single-Head：从 X 一直算到 O

现在用一个只有两个 positions 的例子，把 Step 11.2–11.5 串起来。

取：

```text
T = 2
C = 2
d_k = 2
d_v = 2
```

为了让手算最清楚，输入设为：

$$
X=
\begin{bmatrix}
1&0\\
0&1
\end{bmatrix}.
$$

取三套教学 projection matrices：

$$
W_Q=
\begin{bmatrix}
1&0\\
0&2
\end{bmatrix},
\qquad
W_K=
\begin{bmatrix}
1&1\\
1&0
\end{bmatrix},
$$

$$
W_V=
\begin{bmatrix}
2&0\\
0&4
\end{bmatrix}.
$$

因为 X 是 identity matrix，所以：

$$
Q=XW_Q
=
\begin{bmatrix}
1&0\\
0&2
\end{bmatrix},
$$

$$
K=XW_K
=
\begin{bmatrix}
1&1\\
1&0
\end{bmatrix},
$$

$$
V=XW_V
=
\begin{bmatrix}
2&0\\
0&4
\end{bmatrix}.
$$

接下来只用前四小节已经学过的公式继续。


## 4. Tiny Single-Head：Scores → Scaling → Softmax

### Raw compatibility scores

$$
QK^T
=
\begin{bmatrix}
1&0\\
0&2
\end{bmatrix}
\begin{bmatrix}
1&1\\
1&0
\end{bmatrix}^{T}
=
\begin{bmatrix}
1&1\\
2&0
\end{bmatrix}.
$$

因为：

$$
d_k=2,
$$

所以：

$$
\sqrt{d_k}=\sqrt2.
$$

Scaled scores：

$$
\tilde S
=
\frac{QK^T}{\sqrt2}
=
\begin{bmatrix}
1/\sqrt2&1/\sqrt2\\
\sqrt2&0
\end{bmatrix}.
$$

### Row 0

第一行两个 logits 完全相同，因此 softmax 必然是：

$$
A_{0,:}=[0.5,0.5].
$$

### Row 1

第二行：

$$
[\sqrt2,0]
\approx
[1.414,0].
$$

Softmax 约为：

$$
A_{1,:}
\approx
[0.804,0.196].
$$

所以 attention matrix：

$$
\boxed{
A
\approx
\begin{bmatrix}
0.500&0.500\\
0.804&0.196
\end{bmatrix}
}.
$$

解释：

- Query position 0 对两个 Keys 一样匹配，因此平均取两边的 Value；
- Query position 1 更偏向 Key position 0，所以后面会更多取 `v_0` 的 payload。


## 5. Tiny Single-Head：真正执行 `A @ V`

Value matrix：

$$
V=
\begin{bmatrix}
2&0\\
0&4
\end{bmatrix}.
$$

### Output position 0

$$
o_0
=
0.5[2,0]
+
0.5[0,4].
$$

所以：

$$
\boxed{o_0=[1,2]}.
$$

### Output position 1

使用更精确的近似：

$$
A_{1,:}
\approx
[0.8044,0.1956].
$$

于是：

$$
o_1
=
0.8044[2,0]
+
0.1956[0,4]
$$

$$
\approx
[1.6088,0.7824].
$$

因此最终：

$$
\boxed{
O=AV
\approx
\begin{bmatrix}
1.0000&2.0000\\
1.6088&0.7824
\end{bmatrix}
}.
$$

现在可以真正看到 Self-Attention 在做什么：

> 每个 position 的输出，都不是只保留自己的原始 Value，而是按照它自己的 attention distribution，从整条 sequence 的 Values 中重新组合出一个 **context-dependent representation（依赖上下文的新表示）**。


In [ ]:
import math
import mlx.core as mx

X = mx.array([
    [1.0, 0.0],
    [0.0, 1.0],
])  # (T,C)

W_Q = mx.array([
    [1.0, 0.0],
    [0.0, 2.0],
])  # (C,d_k)

W_K = mx.array([
    [1.0, 1.0],
    [1.0, 0.0],
])  # (C,d_k)

W_V = mx.array([
    [2.0, 0.0],
    [0.0, 4.0],
])  # (C,d_v)

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

d_k = Q.shape[-1]
raw_scores = Q @ K.T
scaled_scores = raw_scores / math.sqrt(d_k)
A = mx.softmax(scaled_scores, axis=-1)
O = A @ V

print('Q:', Q)
print('K:', K)
print('V:', V)
print('raw scores:', raw_scores)
print('scaled scores:', scaled_scores)
print('A:', A)
print('row sums:', mx.sum(A, axis=-1))
print('O:', O)
print('O shape:', O.shape)


## 6. 加回 Batch：完整 Single-Head Shape Flow

真实 batch 中：

$$
X:(B,T,C).
$$

Q/K/V projections：

$$
Q:(B,T,d_k),
\quad
K:(B,T,d_k),
\quad
V:(B,T,d_v).
$$

Compatibility：

$$
QK^T:(B,T,T).
$$

Scaling 与 softmax：

$$
A
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
:(B,T,T).
$$

Value retrieval：

$$
\boxed{
O=AV
:(B,T,d_v)
}.
$$

因此完整 shape chain：

```text
X                  (B,T,C)
├─> Q              (B,T,d_k)
├─> K              (B,T,d_k)
└─> V              (B,T,d_v)

Q @ K^T            (B,T,T)
/ sqrt(d_k)        (B,T,T)
softmax            (B,T,T) = A

A @ V              (B,T,d_v) = O
```

这里每个 output position 仍然对应原来的一个 Query position，所以 T axis 保留下来；变化的是每个 position 的 representation 已经混入了其他 positions 的 Value information。


## 7. 到这里，一个 Tiny Single-Head Self-Attention 已经闭环

把 Step 11.1–11.5 连起来：

```text
为什么分 Q/K/V
↓
Linear projections
↓
QK^T compatibility scores
↓
/ sqrt(d_k)
↓
softmax over Keys
↓
attention weights A
↓
A @ V
↓
context-dependent output O
```

数学上可以压缩成：

$$
\boxed{
O
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
V
}.
$$

但课程刻意不是从这条最终公式开始背，而是用了整个 Step 11 把每一个符号为什么存在逐步推出来。

### 现在还没有什么？

这个 Step 11 的 tiny head 仍然是 **general self-attention**：

- 没有 causal mask；
- 没有限制一个 position 只能看自己和过去；
- 没有进入 autoregressive next-token constraint；
- 没有 Multi-Head Attention；
- 没有 output projection / residual / Transformer block。

这些都不属于本节。

### Step 11.5 材料边界

- Attention matrix A 决定“从每个 position 的 Value 取多少”，Value tensor V 承载真正被取回的 payload。
- 单个输出 `o_i=sum_j A_(i,j)v_j`，是所有 Value vectors 的 attention-weighted sum。
- Batch shape：`A:(B,T,T)`、`V:(B,T,d_v)`，所以 `O=A@V:(B,T,d_v)`。
- 输出 width 跟随 `d_v`，而不是 `d_k`。
- tiny example 已完整走过 `X -> Q/K/V -> QK^T -> scaling -> softmax -> A@V`。
- O 中每个 position 都成为 context-dependent representation，但 general self-attention 还没有 causal constraint。
- 本轮只准备 Step 11.5；不进入 Step 12.1。
